# FSR v2 prod monitoring

This notebook is for quick production checks of the backfill, metadata job, and chunk job. It is read-only, and it is designed to help monitor queue health and table parity.

Use it while a backfill is in flight, especially when checking whether the process is moving, stalled, or incorrectly including non-target years.

In [ ]:
%sql
USE CATALOG vaip;

-- Confirm the prod FSR v2 tables are present
SHOW TABLES IN ai_std_con_field_service_report;

## 1) Metadata and queue status

This is the first check to answer: are we writing all docs, and are we filtering out pre-2016 / post-2016 docs as intended?

In [ ]:
%sql
SELECT
  metadata_status,
  chunk_status,
  COUNT(*) AS docs
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2
GROUP BY 1, 2
ORDER BY 1, 2;

SELECT
  COUNT(*) AS total_docs,
  SUM(CASE WHEN metadata_status = 'pending' THEN 1 ELSE 0 END) AS p1_pending,
  SUM(CASE WHEN metadata_status = 'completed' THEN 1 ELSE 0 END) AS p1_completed,
  SUM(CASE WHEN metadata_status = 'failed' THEN 1 ELSE 0 END) AS p1_failed,
  SUM(CASE WHEN metadata_status = 'date_filtered' THEN 1 ELSE 0 END) AS p1_date_filtered,
  SUM(CASE WHEN chunk_status = 'pending' THEN 1 ELSE 0 END) AS p2_pending,
  SUM(CASE WHEN chunk_status = 'in_progress' THEN 1 ELSE 0 END) AS p2_in_progress,
  SUM(CASE WHEN chunk_status = 'completed' THEN 1 ELSE 0 END) AS p2_completed,
  SUM(CASE WHEN chunk_status = 'failed' THEN 1 ELSE 0 END) AS p2_failed
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2;

## 2) Year filter / post-2016 logic

The v2 pipeline uses a year gate. The code sets the year filter via `FSR_V2_MIN_DOC_YEAR` and optional `FSR_V2_MAX_DOC_YEAR`. In the active metadata job, docs outside the range are marked `date_filtered`, not `completed`.

The production gate is effectively: keep docs from the configured min year onward, and optionally stop at the max year.

In [ ]:
%sql
-- Docs in the allowed year window and status counts
SELECT
  metadata_status,
  COUNT(*) AS docs
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2
WHERE doc_year IS NOT NULL
GROUP BY 1
ORDER BY 1;

-- Any completed docs that fall before the configured minimum year should be zero
SELECT COUNT(*) AS completed_before_min_year
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2
WHERE metadata_status = 'completed'
  AND doc_year IS NOT NULL
  AND doc_year < 2016;

-- Any completed docs at or after max year (if max year is configured) should be zero or limited
SELECT COUNT(*) AS completed_at_or_after_max_year
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2
WHERE metadata_status = 'completed'
  AND doc_year IS NOT NULL
  AND doc_year >= 2016;

## 3) Completed record % and table parity

The key operational question is: how many docs are truly completed vs. pending, failed, or date-filtered?

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN metadata_status = 'completed' THEN 1 ELSE 0 END) AS completed_rows,
  SUM(CASE WHEN metadata_status = 'pending' THEN 1 ELSE 0 END) AS pending_rows,
  SUM(CASE WHEN metadata_status = 'failed' THEN 1 ELSE 0 END) AS failed_rows,
  SUM(CASE WHEN metadata_status = 'date_filtered' THEN 1 ELSE 0 END) AS date_filtered_rows,
  ROUND(100.0 * SUM(CASE WHEN metadata_status = 'completed' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS completed_pct
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2;

-- A quick document count check
SELECT COUNT(DISTINCT document_id) AS distinct_documents
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2;

## 4) Chunk and map coverage checks

These checks confirm whether chunking and equipment mapping are keeping pace with metadata.

In [ ]:
%sql
SELECT
  'metadata' AS table_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT document_id) AS distinct_docs
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2
UNION ALL
SELECT
  'chunks',
  COUNT(*),
  COUNT(DISTINCT document_id)
FROM vaip.ai_std_con_field_service_report.fsr_chunks_v2
UNION ALL
SELECT
  'map',
  COUNT(*),
  COUNT(DISTINCT document_id)
FROM vaip.ai_std_con_field_service_report.fsr_document_equipment_map_v2;

-- Compare how many completed metadata docs have chunk rows
SELECT
  COUNT(DISTINCT m.document_id) AS completed_metadata_docs,
  COUNT(DISTINCT c.document_id) AS chunked_docs,
  ROUND(100.0 * COUNT(DISTINCT c.document_id) / NULLIF(COUNT(DISTINCT m.document_id), 0), 2) AS chunk_coverage_pct
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2 m
LEFT JOIN vaip.ai_std_con_field_service_report.fsr_chunks_v2 c
  ON c.document_id = m.document_id
WHERE m.metadata_status = 'completed';

-- Compare metadata docs to map rows
SELECT
  COUNT(DISTINCT m.document_id) AS completed_metadata_docs,
  COUNT(DISTINCT map_t.document_id) AS mapped_docs,
  ROUND(100.0 * COUNT(DISTINCT map_t.document_id) / NULLIF(COUNT(DISTINCT m.document_id), 0), 2) AS map_coverage_pct
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2 m
LEFT JOIN vaip.ai_std_con_field_service_report.fsr_document_equipment_map_v2 map_t
  ON map_t.document_id = m.document_id
WHERE m.metadata_status = 'completed';

## 5) Current backfill pulse

Use this to spot whether processing is moving or stalled.

In [ ]:
%sql
SELECT
  COUNT(*) AS total_docs,
  SUM(CASE WHEN metadata_status = 'completed' AND chunk_status = 'pending' THEN 1 ELSE 0 END) AS completed_but_chunk_pending,
  SUM(CASE WHEN chunk_status = 'in_progress' THEN 1 ELSE 0 END) AS chunk_in_progress,
  SUM(CASE WHEN chunk_status = 'completed' THEN 1 ELSE 0 END) AS chunk_completed,
  SUM(CASE WHEN metadata_status = 'date_filtered' THEN 1 ELSE 0 END) AS date_filtered_docs
FROM vaip.ai_std_con_field_service_report.fsr_metadata_v2;